<a href="https://colab.research.google.com/github/nkrao8506/learn/blob/main/openenv_smapleipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd


In [2]:
# CNETRAL TENDENCY
def central_tendency(data):
    arr = np.array(data, dtype=float)
    vals, counts = np.unique(data, return_counts=True)
    sorted_arr = np.sort(arr)
    trim = int(0.1 * len(sorted_arr))
    return {
        'mean': np.mean(arr),
        'median': np.median(arr),
        'mode': vals[np.argmax(counts)],
        'geometric_mean': np.exp(np.mean(np.log(arr))) if np.all(arr > 0) else None,
        'trimmed_mean': np.mean(sorted_arr[trim:-trim]) if len(sorted_arr) > 10 else np.mean(arr)
    }

In [3]:
# FIND-S AND CANDIDATE ELIMINATION
class ConceptLearning:
    def __init__(self, attrs): self.attrs, self.n = attrs, len(attrs)
    def _match(self, h, x): return all(a == '?' or a == b for a, b in zip(h, x))

    def find_s(self, ex):
        h = ['∅'] * self.n
        for x, y in ex:
            if y: h = [x[i] if h[i] == '∅' else h[i] if h[i] == x[i] else '?' for i in range(self.n)]
        return h

    def candidate_elimination(self, ex):
        S, G = ['∅'] * self.n, [['?'] * self.n]
        for x, y in ex:
            if y:
                G = [g for g in G if self._match(g, x)]
                S = [x[i] if S[i] == '∅' else S[i] if S[i] == x[i] else '?' for i in range(self.n)]
                G = [g for g in G if self._match(g, S) or g == S]
            else:
                if self._match(S, x):
                    G = [g[:i] + [v] + g[i+1:] for g in G if self._match(g, x)
                         for i, (gv, xv) in enumerate(zip(g, x)) if gv == '?'
                         for v in self.attrs[i] if v != xv]
                    G = [g for g in G if not any(self._match(o, g) and o != g for o in G)]
        return S, G

In [4]:
# BACKPROPOGATION
class NeuralNetwork:
    def __init__(self, layers, lr=0.01):
        self.lr, self.W, self.b = lr, [np.random.randn(layers[i], layers[i+1])*0.01 for i in range(len(layers)-1)], [np.zeros((1, layers[i+1])) for i in range(len(layers)-1)]
    def _sig(self, x): return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    def _dsig(self, x): return x * (1 - x)
    def forward(self, X):
        self.a = [X]
        for w, b in zip(self.W, self.b): self.a.append(self._sig(self.a[-1] @ w + b))
        return self.a[-1]
    def train(self, X, y, epochs=1000):
        for _ in range(epochs):
            self.forward(X)
            d = [(self.a[-1] - y) * self._dsig(self.a[-1])]
            for i in range(len(self.W)-1, 0, -1): d.append((d[-1] @ self.W[i].T) * self._dsig(self.a[i]))
            d.reverse()
            for i in range(len(self.W)):
                self.W[i] -= self.lr * self.a[i].T @ d[i] / X.shape[0]
                self.b[i] -= self.lr * np.sum(d[i], axis=0, keepdims=True) / X.shape[0]
    def predict(self, X): return self.forward(X)

In [5]:
#K-MEANS CLUSTERING
class KMeans:
    def __init__(self, k=3, max_iters=100): self.k, self.max_iters = k, max_iters
    def fit(self, X):
        self.c = X[np.random.choice(X.shape[0], self.k, replace=False)]
        for _ in range(self.max_iters):
            self.labels_ = np.argmin(np.linalg.norm(X[:, None] - self.c, axis=2), axis=1)
            new_c = np.array([X[self.labels_ == i].mean(axis=0) if sum(self.labels_==i) else self.c[i] for i in range(self.k)])
            if np.allclose(self.c, new_c): break
            self.c = new_c
        return self
    def predict(self, X): return np.argmin(np.linalg.norm(X[:, None] - self.c, axis=2), axis=1)

In [6]:
# KNN
class KNN:
    def __init__(self, k=3): self.k = k
    def fit(self, X, y): self.X, self.y = X, y; return self
    def predict(self, X):
        # Broadcasts distance calculations over the entire array for extreme conciseness
        return np.array([np.bincount(self.y[np.argsort(np.linalg.norm(self.X - x, axis=1))[:self.k]]).argmax() for x in X])

In [7]:
# DECISION TREES
class DecisionTree:
    def __init__(self, max_depth=5): self.md = max_depth
    def fit(self, X, y): self.tree = self._build(X, y); return self
    def _build(self, X, y, d=0):
        if d == self.md or len(set(y)) == 1: return np.bincount(y).argmax()
        e = lambda y: -sum(p*np.log2(p) for p in np.bincount(y)/len(y) if p)
        best = max(((i, t, e(y) - sum(len(y[m])/len(y)*e(y[m]) for m in (X[:, i] <= t, X[:, i] > t)))
                    for i in range(X.shape[1]) for t in np.unique(X[:, i])), key=lambda x: x[2], default=(None, None, 0))
        if best[2] == 0: return np.bincount(y).argmax()
        m = X[:, best[0]] <= best[1]
        return (best[0], best[1], self._build(X[m], y[m], d+1), self._build(X[~m], y[~m], d+1))
    def _pred(self, x, n): return n if isinstance(n, (int, np.int64)) else self._pred(x, n[2] if x[n[0]] <= n[1] else n[3])
    def predict(self, X): return np.array([self._pred(x, self.tree) for x in X])

In [8]:
# TO RUN ALL CELLS FOR OUTPUT

def run_all_tests():
    print("="*50 + "\n1. CENTRAL TENDENCY\n" + "="*50)
    data = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
    print(f"Data: {data}\nResult: {central_tendency(data)}\n")

    print("="*50 + "\n2. FIND-S AND CANDIDATE ELIMINATION\n" + "="*50)
    attrs = [['sunny', 'cloudy', 'rainy'], ['warm', 'cold'], ['high', 'normal']]
    ex = [(['sunny', 'warm', 'high'], 1),
          (['sunny', 'warm', 'normal'], 1),
          (['rainy', 'cold', 'high'], 0)]
    clf = ConceptLearning(attrs)
    print(f"Find-S Hypothesis: {clf.find_s(ex)}")
    S, G = clf.candidate_elimination(ex)
    print(f"Candidate Elimination -> S: {S}, G: {G}\n")

    print("="*50 + "\n3. BACKPROPAGATION (XOR Problem)\n" + "="*50)
    np.random.seed(42)
    X = np.array([[0.8, 0], [0.991, 1], [1.001, 0], [1.29, 1]])
    y = np.array([[0], [1], [1], [0]])
    nn = NeuralNetwork([2, 4, 1], lr=0.5)
    nn.train(X, y, epochs=2000)
    print(f"Predictions:\n{nn.predict(X).round(3)}\n")

    print("="*50 + "\n4. K-MEANS CLUSTERING\n" + "="*50)
    np.random.seed(42)
    X_km = np.vstack([np.random.randn(20, 2), np.random.randn(20, 2) + 5, np.random.randn(20, 2) + [0, 5]])
    kmeans = KMeans(k=3).fit(X_km)
    print(f"Centroids:\n{kmeans.c}")
    print(f"Labels distribution: {np.bincount(kmeans.labels_)}\n")

    print("="*50 + "\n5. KNN ALGORITHM\n" + "="*50)
    X_knn = np.array([[1, 2], [2, 3], [3, 4], [6, 7], [7, 8], [8, 9]])
    y_knn = np.array([0, 0, 0, 1, 1, 1])
    X_test = np.array([[2, 2], [7, 7]])
    knn = KNN(k=3).fit(X_knn, y_knn)
    print(f"Test points: {X_test.tolist()}")
    print(f"Predictions: {knn.predict(X_test)}\n")

    print("="*50 + "\n6. DECISION TREE\n" + "="*50)
    X_dt = np.array([[2.5], [3.5], [4.5], [5.5], [6.5], [7.5]])
    y_dt = np.array([0, 0, 0, 1, 1, 1])
    dt = DecisionTree(max_depth=3).fit(X_dt, y_dt)
    print(f"Training data: {X_dt.flatten().tolist()}")
    print(f"Predictions: {dt.predict(X_dt)}\n")

# Execute the tests
run_all_tests()

1. CENTRAL TENDENCY
Data: [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
Result: {'mean': np.float64(55.0), 'median': np.float64(55.0), 'mode': np.int64(10), 'geometric_mean': np.float64(45.28728688116765), 'trimmed_mean': np.float64(55.0)}

2. FIND-S AND CANDIDATE ELIMINATION
Find-S Hypothesis: ['sunny', 'warm', '?']
Candidate Elimination -> S: ['sunny', 'warm', '?'], G: [['?', '?', '?']]

3. BACKPROPAGATION (XOR Problem)
Predictions:
[[0.5]
 [0.5]
 [0.5]
 [0.5]]

4. K-MEANS CLUSTERING
Centroids:
[[-0.17610301 -0.26117067]
 [ 4.95886859  5.07461057]
 [ 0.08577856  5.08608253]]
Labels distribution: [20 19 21]

5. KNN ALGORITHM
Test points: [[2, 2], [7, 7]]
Predictions: [0 1]

6. DECISION TREE
Training data: [2.5, 3.5, 4.5, 5.5, 6.5, 7.5]
Predictions: [0 0 0 1 1 1]

